# Sistemas Inteligentes

## Curso académico 2024-2025

### Laboratorio 2: Búsqueda Metaheurística

#### Instructores

* Juan Carlos Alfaro Jiménez: JuanCarlos.Alfaro@uclm.es
* María Julia Flores Gallego: Julia.Flores@uclm.es
* Ismael García Varea: Ismael.Garcia@uclm.es
* Adrián Rodríguez López: Adrian.Rodriguez18@alu.uclm.es

## Estaciones de Servicio y Energía: Encontrando la Configuración Óptima

## 1. Introducción

¡Noticias emocionantes! El **Ministerio de Transporte y Movilidad Sostenible** ha quedado muy impresionado con las soluciones desarrolladas en nuestro primer trabajo, la Práctica 1. Están particularmente interesados en implementar estos algoritmos en la planificación de rutas de vehículos autónomos, con A* como el método MÁS efectivo para identificar el camino óptimo de manera eficiente. Para avanzar en este proyecto, el Ministerio tiene como objetivo establecer estratégicamente estaciones de servicio en áreas urbanas para apoyar su flota de vehículos autónomos. Estas estaciones funcionarán como centros de flota y proporcionarán servicios esenciales para los vehículos.

Para lograr esto, han solicitado **nuestra experiencia técnica para determinar la distribución óptima de estas estaciones** en los mapas de la ciudad. Sin embargo, no todas las intersecciones son elegibles como ubicación de una estación; el Ministerio ha preseleccionado intersecciones candidatas basándose en criterios específicos establecidos por sus equipos técnicos y administrativos. Su principal enfoque es la sostenibilidad y el acceso equitativo, con el objetivo de garantizar que todos los ciudadanos estén razonablemente cerca de una estación de servicio. Entre estos puntos seleccionados, solo se elegirá un número específico. Para facilitar nuestra tarea de determinar cuáles deben ser, han proporcionado datos sobre la cobertura poblacional de cada intersección candidata, lo que nos permite tener en cuenta tanto el acceso como la cobertura en nuestra estrategia de distribución.

El objetivo principal es garantizar un acceso eficiente a la máxima población posible, manteniendo una distribución equilibrada en toda la ciudad, una consideración vital para un sistema de transporte completamente autónomo.

### 1.1 Objetivos del Laboratorio

En esta práctica, aplicaremos técnicas de búsqueda metaheurística para resolver problemas de optimización combinatoria.

El primer objetivo es comprender la tarea y formularla desde la perspectiva de la búsqueda metaheurística. Implementaremos al menos dos algoritmos:

* **Búsqueda Aleatoria**, como un punto de partida básico que generará múltiples soluciones, evaluará cada una y devolverá la mejor.

* **Algoritmo Genético**, que permitirá configurar varios parámetros, como el tamaño de la población, la tasa de mutación y el número de generaciones, entre otros.


Además, para la evaluación no-continua se tendrá que implementar la Ascensión de Cplinas o Hill Cilimbing (obligatoriamente), y opcionalmente implementar el algoritmo **Iterated Local Search** (ILS), ambos explicados en el Tema 7.

A continuación, analizaremos y compararemos el rendimiento de estos algoritmos ejecutándolos en instancias de problemas de diferente complejidad.

Esperamos que esta práctica te ayude a profundizar en tu comprensión de las técnicas metaheurísticas y te anime a considerar cómo se pueden aplicar en problemas reales de optimización combinatoria.

**¡Buena suerte!**

## 2. Descripción del Problema

### 2.1 Problemas de Entrada

Cada escenario se proporcionará en un archivo en formato `json` que contiene la siguiente información, con el formato de un diccionario cuyas claves son:

* `address`: La dirección utilizada.
* `distance`: Radio máximo utilizado para definir intersecciones y segmentos alrededor de la dirección.
* `intersections`: Una lista de diccionarios con información sobre las intersecciones.
* `segments`: Una lista de diccionarios con información sobre los segmentos, que representan las calles entre dos intersecciones.
* `candidates`: Una lista de pares (identificador, población) que contiene las intersecciones candidatas. Notad que los identificadores en esta lista deben estar incluidos en la lista de intersecciones.
* `number_stations`: El número de estaciones de servicio que se deben ubicar, que no debe superar el número de candidatos.

Cada diccionario en `intersections` incluye tres claves:

* `identifier`: Identificador de la intersección
* `longitude`: Longitud de la intersección
* `latitude`: Latitud de la intersección

Cada diccionario en `segments` incluye cuatro claves:

* `origin`: Intersección de origen
* `destination`: Intersección de destino
* `distance`: Distancia entre las dos intersecciones
* `speed`: Velocidad máxima permitida entre las dos intersecciones

**IMPORTANTE**: `initial` y `final` ya no están incluidos en el archivo JSON, ya que no son necesarios. Durante la evaluación de una posible configuración, estos puntos iniciales y finales cambiarán varias veces. Esto puede requerir algunos ajustes en el código de la Práctica 1 para ejecutar A*. Estos cambios deben estar claramente indicados (tu código debe coincidir con el de la Práctica 1, excepto por estos cambios) y discutidos en la memoria de prácticas.

### 2.2. Ejemplo ilustrativo

Un posible ejemplo de este problema podría ser el que se muestra en la siguiente imagen, que representa una parte de la ciudad de Albacete:

![title](sample-problems-lab2/toy/example.png)

En este caso, el número de estaciones de servicio de vehículos que se deben ubicar es 4, entre las 15 intersecciones candidatas representadas con puntos azules (etiquetadas con la población cubierta). Una posible solución se representa con puntos verdes.

---

##### Nota:

* El archivo que contiene la imagen debe guardarse en la ruta indicada en el código de esta celda.

---

### 2.3 Definición formal del problema

Necesitamos elegir $s$ estaciones de entre $c$ intersecciones candidatas o elegibles, con $s<c$. Por lo tanto, nuestro objetivo es decidir en cuál de estas $c$ intersecciones candidatas debemos ubicar las $s$ estaciones de servicio de vehículos, de manera que se minimice el tiempo promedio de viaje que cada habitante tarda desde su hogar hasta la estación más cercana. Si denotamos por $S$ al vector de tamaño $s$ que contiene las intersecciones en las que se ubican las estaciones de vehículos y por $C$ al vector de intersecciones candidatas que contiene el par (id, pop) para cada intersección candidata, entonces formalmente, queremos resolver el siguiente problema de optimización:

$$
S^* = \arg\min_{S} \frac{1}{\sum_{i=0}^{c-1} C[i].pop} \cdot \min_{j=0,\dots,s-1} \left\{\sum_{i=0}^{c-1} \; C[i].pop \cdot time(C[i].id,S[j])\right\}
$$

donde:
- $C[i].pop$ representa la población (número de habitantes) cubierta por la intersección candidata $i$.
- $C[i].id$ es el identificador de la intersección candidata $i$.
- $time(A,B)$ representa el menor tiempo real para viajar desde la intersección $A$ hasta la intersección $B$.

Las siguientes consideraciones deben tenerse en cuenta respecto a la expresión anterior:
- Estamos tratando con un problema de minimización.
- La cardinalidad del espacio de búsqueda es:

$$
\binom{c}{s} = \frac{c!}{(c-s)!s!}
$$

por ejemplo, si tenemos 20 intersecciones elegibles y 4 estaciones de vehículos, el número de soluciones posibles es 210, no demasiadas; pero si tenemos 100 candidatos y 10 estaciones, entonces el número de soluciones posibles es $1.7\times10^{13}$ ($5.3\times 10^{20}$ con 20 estaciones).

## 3. Desarrollo de la práctica

Antes de implementar los algoritmos, primero debes considerar definir los elementos básicos en este tipo de problemas, a saber:

- Una representación conveniente para las soluciones (configuraciones, cromosomas, individuos, ...) del problema que se utilizarán en los algoritmos de optimización combinatoria. Piensa detenidamente en las distintas opciones y tendrás que discutirlas en el informe de la tarea.

- Implementar un mecanismo de evaluación para gestionar las evaluaciones realizadas por los algoritmos de optimización combinatoria. A continuación, se detallará cómo debe realizarse la evaluación.

- Notas importantes:
    - En el caso de que A* no devuelva ninguna solución (coste = inf), reemplazad este valor por un número muy alto en comparación con el tiempo máximo en nuestro problema. Reflexiona sobre la necesidad de esto y discútelo en el informe.
    - Podéis aprovechar el mecanismo de evaluación para guardar algunos cálculos, recopilar estadísticas e imprimir los resultados.
    - Tened en cuenta que esta tarea requiere que ya hayas resuelto la Práctica 1, y necesitarás reutilizar el código implementado para resolver esta práctica.
   
Tendréis que resolver muchos problemas similares a los de la Práctica 1. Los mapas serán los mismos, pero los problemas necesitan incorporar nueva información, que es la lista de intersecciones candidatas y, para cada una de ellas, la población que cubren. El número de estaciones que se deben ubicar también se indica en el problema como `number_stations`.


### 3.1 Evaluación de una solución

Dada una instancia específica del problema a resolver, y asumiendo que $C$ denota su lista de intersecciones candidatas, el valor de cada posible solución $S$ debe calcularse como:

$$value(S) = \frac{1}{\sum_{i=0}^{c-1} C[i].pop} \cdot \min_{j=0,\dots,s-1} \left\{\sum_{i=0}^{c-1} \; C[i].pop \cdot time(C[i].id,S[j])\right\}$$

de acuerdo con la fórmula presentada en la sección 2.2.

## 4. Plan de trabajo

### 4.1. Tareas

* Transferid y adaptad vuestro código de la Práctica 1 para resolver búsquedas con A* que necesitaréis aquí:
    * Reutilizad la mayor parte del código necesario de vuestra Práctica 1.
    * Describid qué se ha modificado, por qué y cómo.

* Procesad los nuevos archivos JSON y guardad el problema de acuerdo con lo siguiente:
    * Además de las clases de búsqueda (Problem_2, State, Action, ...), deberéis trabajar con las intersecciones candidatas.
    * Construid un mecanismo capaz de almacenar y recuperar las intersecciones candidatas y la población asociada a cada una.

* Representación de una posible configuración:
    * Entre las vistas en el Tema 6, encontrad la representación más adecuada para este problema y adaptadla, teniendo en cuenta que cada problema tiene valores distintos para el número total de intersecciones, intersecciones elegibles y el número solicitado de estaciones.
    * Conectadla a un método adecuado para evaluar cada una considerando las indicaciones mencionadas anteriormente en el punto 3.2.

* Implementación de algoritmos:
    * Implementad, al menos, los dos algoritmos requeridos (Búsqueda Aleatoria y un Algoritmo Genético — GA). Tened en cuenta que para la evaluación no continua, también deberéis agregar Hill Climbing y, opcionalmente, ILS.
    * En el GA, aseguraos de haber implementado la generación de una población junto con los elementos principales dentro del bucle principal: selección, cruce, mutación y combinación de generaciones.

* Experimentación y análisis:
    * Los parámetros que se puedan ajustar deben ser explorados adecuadamente, también en relación con los problemas dados (dimensionalidad, complejidad, etc.).
    * También deberéis estudiar el rendimiento resultante en términos de desempeño, convergencia, número de generaciones, etc.
    * Comparad la Búsqueda Aleatoria y el Algoritmo Genético, asegurándoos de obtener resultados consistentes.

* Informe:
    * Redactad un informe detallando el proceso seguido, las estrategias implementadas y los resultados obtenidos, junto con gráficos y comparaciones visuales.


### 4.2. Evaluación de la práctica

En la modalidad de **evaluación continua**, la evaluación de la práctica se realizará a través de un examen individual en el que se tendrá en cuenta lo siguiente:

* Definición e implementación correcta de la representación de la configuración y de la función de evaluación: 25%
* Implementación correcta del algoritmo genético: 50%, que cubre
    * El bucle general para las generaciones es correcto: 10%
    * Los distintos operadores están correctamente diseñados y codificados: 40%
* Eficiencia y optimización: 15%
* Experimentación realizada y análisis de resultados: 10%

Es necesario que la Práctica 1 esté correctamente integrada y que la Búsqueda Aleatoria funcione de manera consistente para que todos los estudiantes puedan utilizarla como un punto de partida base adecuado.

Todo esto se ponderará según el nivel de conocimiento que el estudiante demuestre sobre la práctica en caso de que el examen sea una entrevista personal.

En la modalidad de **evaluación no continua**, la evaluación se modificará como se indica a continuación:

* Definición e implementación correcta de la representación de la configuración y de la función de evaluación: 15%
* Implementación correcta del algoritmo genético: 40%, que cubre
    * El bucle general para las generaciones es correcto: 7%
    * Los distintos operadores están correctamente diseñados y codificados: 33%
* Implementación correcta del algoritmo de Hill Climbing (obligatorio): 15%
* Implementación correcta del algoritmo ILS (opcional): 5%
* Eficiencia y optimización: 15%
* Experimentación realizada y análisis de resultados: 10%


### 4.3. Fechas importantes

* Fecha límite para entregar el código: **13 de diciembre de 2024**.
* Fecha límite para la entrega del informe: **Final del semestre**.



In [13]:
import json
import queue
import timeit
from abc import abstractmethod, ABC

In [14]:
class Problema:
    def __init__(self, info_json):
        with open(info_json, 'r') as archivo:
            problema = json.load(archivo)

        #self.inicio = problema['initial']
        #self.final = problema['final']
        self.interseccionAccion = {}
        self.interseccionesCoordenadas = {}
        self.velMax = 1
        for interseccion in problema['intersections']:
            identificador = interseccion['identifier']
            self.interseccionesCoordenadas[identificador] = (interseccion['longitude'], interseccion['latitude'])

        for segmento in problema['segments']:
            if segmento['origin'] not in self.interseccionAccion:
                self.interseccionAccion[segmento['origin']] = queue.PriorityQueue()
            self.velMax = max(self.velMax, segmento['speed'])
            self.interseccionAccion[segmento['origin']].put((
                segmento['destination'],
                segmento['distance'] / (segmento['speed'] / 3.6)
            ))
        self.candidatos = tuple(problema['segments']['candidates'])
        self.number_stations = problema['segments']['number_stations']

In [15]:
class Estado:

    def __init__(self, id, longitud, latitud):
        self.id = id
        self.longitud = longitud
        self.latitud = latitud

    def __hash__(self):
        return hash(self.id)

    def __eq__(self, otro):
        return self.id == otro.id

In [16]:
class Nodo:

    def __init__(self, id, longitud, latitud, profundidad=0, padre=None, coste=0.0):
        self.id = id
        self.longitud = longitud
        self.latitud = latitud
        self.estado = Estado(self.id, self.longitud, self.latitud)
        self.profundidad = profundidad
        self.padre = padre
        self.coste = coste

    def __repr__(self):
        padre_id = self.padre.id if self.padre is not None else None
        return f"Nodo(id={self.id}, longitud={self.longitud}, latitud={self.latitud}, padre={padre_id}, coste={self.coste})"

    def __eq__(self, otro):
        return self.id == otro.id

    def __hash__(self):
        return hash(self.estado)

    def __lt__(self, otro):
        return self.id < otro.id

In [17]:
class Heuristica:

    def calculo_heuristica(estado1: Estado, tupla_coordenadas, velMax):
        return abs(estado1.longitud - tupla_coordenadas[0]) + abs(estado1.latitud - tupla_coordenadas[1]) / (velMax / 3.6)
    
    

In [18]:
class Busqueda(ABC):

    def __init__(self):
        self.problema = Problema(archivo_json)
        self.listaAbiertos = None

    @abstractmethod
    def insertarNodo(self, nodo, lista_nodos):
        pass

    @abstractmethod
    def extraerNodo(self, lista_nodos):
        pass

    @abstractmethod
    def vacio(self, lista_nodo):
        pass

    def nodosSucesores(self, nodo):
        sucesores = []
        if nodo.id in self.problema.interseccionAccion:
            pq = self.problema.interseccionAccion[nodo.id]
            while not pq.empty():
                accion = pq.get()
                nodoNuevo = Nodo(accion[0], self.problema.interseccionesCoordenadas[accion[0]][0],
                                 self.problema.interseccionesCoordenadas[accion[0]][1],
                                 nodo.profundidad + 1, nodo, nodo.coste + accion[1])
                sucesores.append(nodoNuevo)
        return sucesores

    def buscar(self, estadoFinal):
        longitud, latitud = self.problema.interseccionesCoordenadas[self.problema.inicio]
        nodo_progenitor = Nodo(self.problema.inicio, longitud, latitud)
        profundidad = 0
        lista_expandidos = set()
        expandidos = 1
        generados = 0
        tiempo_inicio = timeit.default_timer()
        self.listaAbiertos = self.insertarNodo(nodo_progenitor, self.listaAbiertos)
        while not self.vacio(self.listaAbiertos):
            nodo = self.extraerNodo(self.listaAbiertos)
            estado_nodoExpandido = Estado(nodo.id, nodo.longitud, nodo.latitud)
            if estado_nodoExpandido not in lista_expandidos:
                if estado_nodoExpandido.__eq__(estadoFinal):
                    tiempo_final = timeit.default_timer()
                    segundos = tiempo_final - tiempo_inicio
                    self.camino(segundos, expandidos, generados, self.profundidad, nodo, lista_expandidos)
                    return "Retorno del metodo buscar() = Exito"
                nuevosAbiertos = self.nodosSucesores(nodo)
                expandidos += 1
                generados += len(nuevosAbiertos)
                for abierto in nuevosAbiertos:
                    self.profundidad = max(profundidad, abierto.profundidad)
                    self.insertarNodo(abierto, self.listaAbiertos)
                    lista_expandidos.add(nodo)
        return "Retorno del metodo buscar() = Fracaso"

    def camino(self, segundos, expandidos, abiertos, profundidad, nodoExpandido, listaExpantidos):
        nodo = nodoExpandido
        lista = []
        while nodo.padre is not None:
            lista.append([nodo.padre.id, nodo.id, nodo.coste])
            nodo = nodo.padre
        lista = reversed(lista)
        print("Camino")
        for x in lista:
            print(f"{x[0]} ------({  x[2]:<13})-----> {x[1]}") #Coste acumulado

        print("")
        print(f" Tiempo empleado: {segundos:.10f} segundos")
        print("Nodos expandidos: ", expandidos)
        print("  Nodos generados: ", abiertos)
        print("     Profundidad: ", profundidad)
        print("Velocidad máxima: ", self.problema.velMax)
        print("    Nodo destino: ", nodoExpandido)
        print("             Fin: ", self.problema.final)
        print("          Origen: ", self.problema.inicio)
        print("   Tamaño camino:", lista.__sizeof__())
        print("")
        print("")
        # print("Lista expandidos: ", listaExpandidos)

In [19]:
class AEstrella(Busqueda):

    def __init__(self):
        super().__init__()
        self.listaAbiertos = queue.PriorityQueue()

    def insertarNodo(self, nodo, lista_nodos):
        f = Heuristica.calculo_heuristica(nodo.estado,self.problema.interseccionesCoordenadas[self.problema.final],self.problema.velMax)
        f = f + nodo.coste
        lista_nodos.put((f, nodo))
        return lista_nodos

    def extraerNodo(self, lista_nodos):
        return lista_nodos.get()[1]

    def vacio(self, lista_nodos):
        return lista_nodos.empty()

In [20]:
if __name__ == "__main__":
    archivo_json = r"C:\Users\Vlad\OneDrive - Universidad de Castilla-La Mancha\Escritorio\Practica2\sample-problems-lab2\toy\calle_del_virrey_morcillo_albacete_250_3_candidates_15_ns_4.json"
    #archivo_json = r"C:\Users\eduardo\PycharmProjects\S.-Inteligentes-proyecto\calle_de_francisco_5000_3.json"
    #archivo_json = r"C:\Users\eduardo\PycharmProjects\S.-Inteligentes-proyecto\calle_agustina_aroca_albacete_5000_0.json"
    #archivo_json = r"C:\Users\eduardo\PycharmProjects\S.-Inteligentes-proyecto\calle_marila_mariln_500_1.json"
    problema = Problema(archivo_json)
    z = AEstrella()
    print(z.buscar(Estado(z.problema.final,
                    z.problema.interseccionesCoordenadas[z.problema.final][0],
                    z.problema.interseccionesCoordenadas[z.problema.final][1])))



KeyError: 'final'